# DS7 · Datos a escala — Práctica guiada

Caso **TiendaSur**: el equipo de datos recibe cada día los pedidos de la tienda online y debe dejarlos limpios y agregados para negocio. En este notebook construyes, a pequeña escala y sin instalar nada pesado, las piezas que piden las ofertas de ingeniería y ciencia de datos:

1. Motores: **pandas vs Polars vs DuckDB** con un benchmark real de 4 millones de filas.
2. **Parquet** y particionado.
3. Almacén por capas (**bronce → plata → oro**) y modelo en estrella.
4. **dbt**: modelos SQL y tests de datos.
5. **Spark**: equivalencias de API y qué hace por dentro.
6. **Orquestación**: un DAG con reintentos e idempotencia (más el DAG de Airflow como archivo).
7. **Streaming**: un consumidor con ventanas y marca de agua (más el esqueleto de Kafka).

> Todo se escribe en `proyecto-datos-escala/`. Tiempo total en un portátil: menos de 2 minutos.

In [ ]:
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# Todo lo que escribe este notebook va a esta carpeta (está en .gitignore del curso).
BASE = Path("proyecto-datos-escala").resolve()
if BASE.exists():
    shutil.rmtree(BASE)            # empezamos de cero: reproducibilidad
BASE.mkdir()

def cronometrar(f, repeticiones=3):
    """Mejor tiempo de varias ejecuciones (la primera calienta cachés del disco)."""
    mejor, resultado = float("inf"), None
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        resultado = f()
        mejor = min(mejor, time.perf_counter() - t0)
    return mejor, resultado

print("pandas", pd.__version__, "| polars", pl.__version__, "| duckdb", duckdb.__version__, "| pyarrow", pa.__version__)

## 7.1 · Datos: cuatro millones de líneas de pedido

Con 4 millones de filas todavía cabe en la memoria de un portátil, que es justo el punto interesante: **no necesitas un clúster** para notar que pandas se queda corto. Fíjate en cuánto ocupa en memoria.

In [ ]:
# Generamos las líneas de pedido de un año de TiendaSur: 4 millones de filas.
rng = np.random.default_rng(7)
N = 4_000_000
CATEGORIAS = np.array(["café", "té", "menaje", "accesorios"])
CANALES = np.array(["web", "tienda", "app"])
PROVINCIAS = np.array(["Sevilla", "Málaga", "Madrid", "Barcelona", "Valencia", "Bilbao", "Cádiz", "Granada"])

productos = pd.DataFrame({
    "id_producto": np.arange(1, 401),
    "categoria": rng.choice(CATEGORIAS, 400, p=[0.40, 0.15, 0.30, 0.15]),
    "precio": (rng.gamma(4, 4, 400) + 2).round(2),
})

dias = pd.date_range("2025-10-01", "2026-09-30")            # 365 días
meses_de_dia = np.array(dias.strftime("%Y-%m"))
idx_dia = rng.integers(0, len(dias), N)
id_producto = rng.integers(1, 401, N)

pedidos = pd.DataFrame({
    "id_linea": np.arange(N),
    "id_pedido": np.arange(N) // 3,
    "fecha": dias.values[idx_dia],
    "mes": meses_de_dia[idx_dia],
    "id_cliente": rng.integers(1, 200_001, N),
    "id_producto": id_producto,
    "cantidad": rng.integers(1, 5, N),
    "precio_unitario": productos["precio"].to_numpy()[id_producto - 1],
    "canal": rng.choice(CANALES, N, p=[0.55, 0.30, 0.15]),
    "provincia": rng.choice(PROVINCIAS, N),
    "estado": rng.choice(["entregado", "devuelto", "cancelado"], N, p=[0.90, 0.06, 0.04]),
})
print(f"{len(pedidos):,} filas · {pedidos.memory_usage(deep=True).sum() / 1e6:,.0f} MB en la memoria de pandas")
pedidos.head(3)

## 7.2 · Formatos columnares: Parquet

CSV guarda **filas** como texto; Parquet guarda **columnas** comprimidas y con tipos. Dos consecuencias:
- archivos mucho más pequeños y rápidos de leer;
- si tu consulta usa 2 columnas de 10, solo se leen esas 2 (*projection pushdown*).

Además, **particionar** por una columna (aquí `mes`) crea una carpeta por valor; con un filtro por esa columna el motor ni abre las demás carpetas (*partition pruning*).

In [ ]:
# Guardamos el dataset en Parquet particionado por mes (una carpeta por mes).
RUTA_PART = BASE / "pedidos_parquet"
t0 = time.perf_counter()
pq.write_to_dataset(pa.Table.from_pandas(pedidos, preserve_index=False), RUTA_PART, partition_cols=["mes"])
print(f"Escrito en {time.perf_counter() - t0:.1f} s")

def tam_mb(ruta):
    ruta = Path(ruta)
    archivos = [ruta] if ruta.is_file() else [p for p in ruta.rglob("*") if p.is_file()]
    return sum(p.stat().st_size for p in archivos) / 1e6

print("Carpetas de partición:", sorted(p.name for p in RUTA_PART.iterdir())[:3], "...", len(list(RUTA_PART.iterdir())), "en total")
print(f"Parquet particionado: {tam_mb(RUTA_PART):,.0f} MB en disco (vs {pedidos.memory_usage(deep=True).sum() / 1e6:,.0f} MB en memoria)")

In [ ]:
# CSV frente a Parquet con una muestra de 500.000 filas.
muestra = pedidos.sample(500_000, random_state=1).drop(columns=["mes"])
muestra.to_csv(BASE / "muestra.csv", index=False)
muestra.to_parquet(BASE / "muestra.parquet", index=False)          # compresión snappy por defecto
print(f"CSV:     {tam_mb(BASE / 'muestra.csv'):6.1f} MB")
print(f"Parquet: {tam_mb(BASE / 'muestra.parquet'):6.1f} MB")

# Leer TODAS las columnas vs leer solo las 2 que necesito.
t_csv, _ = cronometrar(lambda: pd.read_csv(BASE / "muestra.csv"))
t_pq_todo, _ = cronometrar(lambda: pd.read_parquet(BASE / "muestra.parquet"))
t_pq_2, _ = cronometrar(lambda: pd.read_parquet(BASE / "muestra.parquet", columns=["id_producto", "precio_unitario"]))
print(f"\nLeer CSV completo:          {t_csv:.3f} s")
print(f"Leer Parquet completo:      {t_pq_todo:.3f} s")
print(f"Leer Parquet, 2 columnas:   {t_pq_2:.3f} s   <- el formato columnar solo toca lo que pides")
print("(Con 500.000 filas los tiempos son pequeños; la ventaja crece con el tamaño, con las columnas de texto y en disco/red.)")

# Poda de particiones: con un filtro por mes, ¿cuántos archivos hay que abrir?
dataset = ds.dataset(RUTA_PART, format="parquet", partitioning="hive")
total = len(list(dataset.get_fragments()))
leidos = len(list(dataset.get_fragments(filter=ds.field("mes") == "2026-09")))
print(f"\nFiltro mes = 2026-09: se abren {leidos} de {total} archivos")

## 7.3 · pandas vs Polars vs DuckDB

La misma consulta (ingresos y clientes distintos por mes y categoría, solo pedidos entregados, uniendo con el catálogo) en tres motores. Reglas de un benchmark honesto: mismos datos de entrada, **mismo resultado comprobado**, varias repeticiones y quedarse con el mejor tiempo.

- **pandas** es eager (ejecuta paso a paso) y lee lo que le dices.
- **Polars** es *lazy*: construye un plan, lo optimiza (filtra y proyecta antes de leer) y lo ejecuta en paralelo.
- **DuckDB** es un motor SQL analítico embebido: sin servidor, lee Parquet directamente y también optimiza el plan.

In [ ]:
RUTA_GLOB = str(RUTA_PART / "**" / "*.parquet")
CONSULTA_SQL = f"""
SELECT p.mes, pr.categoria, SUM(p.cantidad * p.precio_unitario) AS ingresos,
       COUNT(DISTINCT p.id_cliente) AS clientes
FROM read_parquet('{RUTA_GLOB}', hive_partitioning = true) AS p
JOIN productos AS pr USING (id_producto)
WHERE p.estado = 'entregado'
GROUP BY p.mes, pr.categoria
ORDER BY p.mes, pr.categoria
"""

def con_pandas():
    df = pd.read_parquet(RUTA_PART)                                     # lee TODAS las columnas
    df = df[df["estado"] == "entregado"].merge(productos[["id_producto", "categoria"]], on="id_producto")
    df["importe"] = df["cantidad"] * df["precio_unitario"]
    df["mes"] = df["mes"].astype(str)
    return (df.groupby(["mes", "categoria"], as_index=False)
              .agg(ingresos=("importe", "sum"), clientes=("id_cliente", "nunique")))

def con_pandas_columnas():
    cols = ["mes", "id_cliente", "id_producto", "cantidad", "precio_unitario", "estado"]
    df = pd.read_parquet(RUTA_PART, columns=cols)                       # proyección: solo lo necesario
    df = df[df["estado"] == "entregado"].merge(productos[["id_producto", "categoria"]], on="id_producto")
    df["importe"] = df["cantidad"] * df["precio_unitario"]
    df["mes"] = df["mes"].astype(str)
    return (df.groupby(["mes", "categoria"], as_index=False)
              .agg(ingresos=("importe", "sum"), clientes=("id_cliente", "nunique")))

def con_polars():
    return (pl.scan_parquet(RUTA_GLOB, hive_partitioning=True)          # perezoso: nada se lee todavía
            .filter(pl.col("estado") == "entregado")
            .join(pl.from_pandas(productos[["id_producto", "categoria"]]).lazy(), on="id_producto")
            .group_by("mes", "categoria")
            .agg((pl.col("cantidad") * pl.col("precio_unitario")).sum().alias("ingresos"),
                 pl.col("id_cliente").n_unique().alias("clientes"))
            .sort("mes", "categoria")
            .collect().to_pandas())

def con_duckdb():
    con = duckdb.connect()
    con.register("productos", productos)
    return con.sql(CONSULTA_SQL).df()

resultados, tiempos = {}, {}
for nombre, f in [("pandas (todo)", con_pandas), ("pandas (columnas)", con_pandas_columnas),
                  ("polars (lazy)", con_polars), ("duckdb (SQL)", con_duckdb)]:
    tiempos[nombre], resultados[nombre] = cronometrar(f)

# 1) ¿Dan lo mismo? Un benchmark sin comprobar el resultado no vale nada.
ref = resultados["duckdb (SQL)"].sort_values(["mes", "categoria"]).reset_index(drop=True)
for nombre, r in resultados.items():
    r = r.sort_values(["mes", "categoria"]).reset_index(drop=True)
    assert np.allclose(r["ingresos"], ref["ingresos"]) and (r["clientes"].to_numpy() == ref["clientes"].to_numpy()).all(), nombre
print("Los cuatro motores devuelven exactamente el mismo resultado.\n")

# 2) Tiempos
tabla = pd.DataFrame({"segundos": tiempos}).round(2)
tabla["veces_mas_rapido_que_pandas"] = (tiempos["pandas (todo)"] / tabla["segundos"]).round(1)
print(tabla)
ref.head(4)

## 7.4 · Almacén por capas y modelo en estrella

- **Bronce:** copia fiel de lo que llega, más metadatos (cuándo, de qué lote). Nunca se modifica: es tu seguro.
- **Plata:** limpio, tipado, sin duplicados; lo que se descarta se guarda con su motivo.
- **Oro:** tablas pensadas para responder preguntas de negocio: un **hecho** (`fct_ventas_dia`, grano día × producto × canal) y sus **dimensiones** (`dim_producto`). Es el *esquema en estrella*.

Clave de diseño: cada paso es **idempotente** (borra lo del día y lo vuelve a insertar). Así puedes reprocesar sin miedo.

In [ ]:
# Arquitectura medallón en DuckDB: bronce (crudo) -> plata (limpio) -> oro (para negocio).
RUTA_DB = BASE / "almacen.duckdb"

def abrir():
    return duckdb.connect(str(RUTA_DB))

def inicializar(con):
    con.execute("CREATE SCHEMA IF NOT EXISTS bronce; CREATE SCHEMA IF NOT EXISTS plata; CREATE SCHEMA IF NOT EXISTS oro;")
    con.execute("""CREATE TABLE IF NOT EXISTS bronce.pedidos (
        id_linea BIGINT, id_pedido BIGINT, fecha DATE, id_cliente INTEGER, id_producto INTEGER,
        cantidad INTEGER, precio_unitario DOUBLE, canal VARCHAR, provincia VARCHAR, estado VARCHAR,
        _lote DATE, _ingestado TIMESTAMP DEFAULT current_timestamp)""")
    con.execute("""CREATE TABLE IF NOT EXISTS plata.pedidos (
        id_linea BIGINT PRIMARY KEY, id_pedido BIGINT, fecha DATE, id_cliente INTEGER, id_producto INTEGER,
        cantidad INTEGER, precio_unitario DOUBLE, canal VARCHAR, provincia VARCHAR, estado VARCHAR)""")
    con.execute("""CREATE TABLE IF NOT EXISTS plata.rechazados (
        id_linea BIGINT, fecha DATE, motivo VARCHAR)""")
    con.execute("""CREATE TABLE IF NOT EXISTS oro.dim_producto AS
        SELECT id_producto, categoria, precio AS precio_lista FROM productos""")
    con.execute("""CREATE TABLE IF NOT EXISTS oro.fct_ventas_dia (
        fecha DATE, id_producto INTEGER, canal VARCHAR, pedidos BIGINT, unidades BIGINT, ingresos DOUBLE)""")

def ensuciar(df, semilla=0):
    """Simula un extracto real: duplicados, nulos, cantidades imposibles y canales mal escritos."""
    r = np.random.default_rng(semilla)
    df = df.copy()
    df.loc[r.random(len(df)) < 0.010, "precio_unitario"] = np.nan                 # 1 % sin precio
    df.loc[r.random(len(df)) < 0.005, "cantidad"] = -1                            # 0,5 % cantidad negativa
    sucios = r.random(len(df)) < 0.05
    df.loc[sucios, "canal"] = df.loc[sucios, "canal"].map(lambda c: r.choice([c.upper(), f" {c} "]))
    return pd.concat([df, df.sample(frac=0.02, random_state=semilla)], ignore_index=True)   # 2 % duplicados

def lote_del_dia(fecha):
    dia = pedidos[pedidos["fecha"] == pd.Timestamp(fecha)].drop(columns=["mes"])
    return ensuciar(dia, semilla=int(pd.Timestamp(fecha).strftime("%Y%m%d")))

print("Día de ejemplo:", len(lote_del_dia("2026-09-01")), "filas (incluye suciedad intencionada)")

In [ ]:
def cargar_bronce(con, fecha, lote):
    """Bronce: tal cual llega, más metadatos. Idempotente: borra el lote antes de insertarlo."""
    con.execute("DELETE FROM bronce.pedidos WHERE _lote = ?", [fecha])
    con.register("lote_tmp", lote)
    con.execute("""INSERT INTO bronce.pedidos (id_linea, id_pedido, fecha, id_cliente, id_producto, cantidad,
                   precio_unitario, canal, provincia, estado, _lote)
                   SELECT id_linea, id_pedido, fecha, id_cliente, id_producto, cantidad, precio_unitario,
                          canal, provincia, estado, CAST(? AS DATE) FROM lote_tmp""", [fecha])
    con.unregister("lote_tmp")

def limpiar_plata(con, fecha):
    """Plata: tipos correctos, sin duplicados, valores normalizados. Lo rechazado se guarda con su motivo."""
    con.execute("DELETE FROM plata.pedidos WHERE fecha = ?", [fecha])
    con.execute("DELETE FROM plata.rechazados WHERE fecha = ?", [fecha])
    con.execute("""INSERT INTO plata.rechazados
        SELECT id_linea, fecha, CASE WHEN precio_unitario IS NULL THEN 'precio nulo' ELSE 'cantidad no positiva' END
        FROM bronce.pedidos WHERE _lote = ? AND (precio_unitario IS NULL OR cantidad <= 0)""", [fecha])
    con.execute("""INSERT INTO plata.pedidos
        SELECT id_linea, id_pedido, fecha, id_cliente, id_producto, cantidad, precio_unitario,
               lower(trim(canal)) AS canal, provincia, estado
        FROM bronce.pedidos
        WHERE _lote = ? AND precio_unitario IS NOT NULL AND cantidad > 0
        QUALIFY row_number() OVER (PARTITION BY id_linea ORDER BY _ingestado DESC) = 1""", [fecha])

def agregar_oro(con, fecha):
    """Oro: hechos agregados con el grano acordado (día x producto x canal)."""
    con.execute("DELETE FROM oro.fct_ventas_dia WHERE fecha = ?", [fecha])
    con.execute("""INSERT INTO oro.fct_ventas_dia
        SELECT fecha, id_producto, canal, COUNT(DISTINCT id_pedido), SUM(cantidad), SUM(cantidad * precio_unitario)
        FROM plata.pedidos WHERE fecha = ? AND estado = 'entregado' GROUP BY ALL""", [fecha])

def validar_calidad(con, fecha, max_rechazo=0.05):
    """Puerta de calidad: si se rechaza demasiado, algo ha ido mal aguas arriba."""
    ok, mal = con.execute("""SELECT (SELECT COUNT(*) FROM plata.pedidos WHERE fecha = ?),
                                    (SELECT COUNT(*) FROM plata.rechazados WHERE fecha = ?)""", [fecha, fecha]).fetchall()[0]
    tasa = mal / max(ok + mal, 1)
    if tasa > max_rechazo:
        raise ValueError(f"{fecha}: {tasa:.1%} de filas rechazadas (límite {max_rechazo:.0%})")
    return tasa

con = abrir()
con.register("productos", productos)
inicializar(con)
for f in ["2026-09-01", "2026-09-02", "2026-09-03"]:
    cargar_bronce(con, f, lote_del_dia(f)); limpiar_plata(con, f); agregar_oro(con, f)
    print(f, "-> tasa de rechazo", f"{validar_calidad(con, f):.2%}")

print(con.sql("""SELECT 'bronce' AS capa, COUNT(*) AS filas FROM bronce.pedidos UNION ALL
                 SELECT 'plata', COUNT(*) FROM plata.pedidos UNION ALL
                 SELECT 'rechazados', COUNT(*) FROM plata.rechazados UNION ALL
                 SELECT 'oro', COUNT(*) FROM oro.fct_ventas_dia""").df())

# Idempotencia: reprocesar un día no duplica nada.
antes = con.sql("SELECT SUM(ingresos) FROM oro.fct_ventas_dia").fetchall()[0][0]
cargar_bronce(con, "2026-09-02", lote_del_dia("2026-09-02")); limpiar_plata(con, "2026-09-02"); agregar_oro(con, "2026-09-02")
despues = con.sql("SELECT SUM(ingresos) FROM oro.fct_ventas_dia").fetchall()[0][0]
assert abs(antes - despues) < 1e-6
print(f"\nReprocesar el 2 de septiembre deja los ingresos igual: {antes:,.2f} € -> {despues:,.2f} €")

# Esquema en estrella: un hecho y sus dimensiones. Los analistas preguntan unidos por claves.
print(con.sql("""SELECT d.categoria, f.canal, ROUND(SUM(f.ingresos), 0) AS ingresos
                 FROM oro.fct_ventas_dia f JOIN oro.dim_producto d USING (id_producto)
                 GROUP BY ALL ORDER BY ingresos DESC LIMIT 5""").df())

## 7.5 · dbt: SQL con estructura, dependencias y tests

dbt no mueve datos: **transforma dentro del almacén**. Escribes `select` en archivos `.sql`; `{{ ref('otro_modelo') }}` declara dependencias (dbt deduce el orden) y los tests (`unique`, `not_null`, `accepted_values`, `relationships`) comprueban los datos. Aquí lo usamos con `dbt-duckdb` sobre el mismo almacén. Primero, los archivos del proyecto:

In [ ]:
DBT = BASE / "tiendasur_dbt"

def escribir(ruta, contenido):
    destino = DBT / ruta
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(contenido.lstrip("\n"))
    print("escrito:", destino.relative_to(BASE))

escribir("dbt_project.yml", """
name: tiendasur
version: "1.0.0"
profile: tiendasur
model-paths: ["models"]
test-paths: ["tests"]
models:
  tiendasur:
    staging:
      +materialized: view
    marts:
      +materialized: table
""")

escribir("profiles.yml", """
tiendasur:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: ../almacen.duckdb
      schema: dbt
      threads: 1
""")

escribir("models/staging/sources.yml", """
version: 2
sources:
  - name: plata
    schema: plata
    tables:
      - name: pedidos
      - name: productos
""")

escribir("models/staging/stg_pedidos.sql", """
-- Una fila por línea de pedido, con el importe ya calculado.
select
    id_linea, id_pedido, fecha, id_cliente, id_producto, cantidad, precio_unitario,
    cantidad * precio_unitario as importe,
    canal, estado
from {{ source('plata', 'pedidos') }}
""")

escribir("models/marts/dim_producto.sql", """
select id_producto, categoria, precio as precio_lista
from {{ source('plata', 'productos') }}
""")

escribir("models/marts/fct_ventas_dia.sql", """
-- Grano: día x producto x canal. Solo pedidos entregados.
select
    fecha, id_producto, canal,
    count(distinct id_pedido) as pedidos,
    sum(cantidad)             as unidades,
    sum(importe)              as ingresos
from {{ ref('stg_pedidos') }}
where estado = 'entregado'
group by 1, 2, 3
""")

escribir("models/marts/schema.yml", """
version: 2
models:
  - name: dim_producto
    columns:
      - name: id_producto
        tests: [unique, not_null]
  - name: fct_ventas_dia
    columns:
      - name: id_producto
        tests:
          - not_null
          - relationships:
              arguments: {to: "ref('dim_producto')", field: id_producto}
      - name: canal
        tests:
          - accepted_values:
              arguments: {values: ['web', 'tienda', 'app']}
      - name: ingresos
        tests: [not_null]
""")
escribir("models/staging/schema.yml", """
version: 2
models:
  - name: stg_pedidos
    columns:
      - name: id_linea
        tests: [unique, not_null]
""")

escribir("tests/ingresos_no_negativos.sql", """
-- Un test singular: devuelve las filas que violan la regla. Si no devuelve ninguna, pasa.
select * from {{ ref('fct_ventas_dia') }} where ingresos < 0
""")

`dbt build` ejecuta modelos y tests en orden de dependencias:

In [ ]:
# DuckDB solo admite un proceso escribiendo a la vez: cerramos nuestra conexión antes de llamar a dbt.
con.execute("CREATE OR REPLACE TABLE plata.productos AS SELECT * FROM productos")
con.close()

DBT_EXE = Path(sys.executable).parent / "dbt"
ENTORNO_DBT = {**os.environ, "DBT_SEND_ANONYMOUS_USAGE_STATS": "false", "DO_NOT_TRACK": "1", "NO_COLOR": "1"}

def dbt(*args):
    """Ejecuta dbt como en la terminal y devuelve (código, líneas relevantes de la salida)."""
    if not DBT_EXE.exists():
        print("dbt no está instalado (pip install dbt-duckdb): saltamos esta celda; los archivos siguen siendo válidos.")
        return None, []
    r = subprocess.run([str(DBT_EXE), *args, "--profiles-dir", ".", "--project-dir", "."],
                       cwd=DBT, capture_output=True, text=True, env=ENTORNO_DBT)
    clave = ("OK created", "PASS", "FAIL", "ERROR", "Done.", "WARN", "Failure")
    limpio = re.sub(r"\x1b\[[0-9;]*m", "", r.stdout + r.stderr)              # quita los colores ANSI
    lineas = [l.split("  ", 1)[1] if l[:2].isdigit() else l for l in limpio.splitlines() if any(k in l for k in clave)]
    return r.returncode, lineas

t0 = time.perf_counter()
codigo, salida = dbt("build")
print(*salida, sep="\n")
print(f"\ncódigo de salida: {codigo} · {time.perf_counter() - t0:.1f} s")

In [ ]:
# Los tests de datos sirven para esto: detectar un dato malo ANTES de que llegue a un dashboard.
con = abrir()
con.execute("INSERT INTO plata.pedidos VALUES (999999999, 1, DATE '2026-09-01', 1, 1, 1, 10.0, 'fax', 'Sevilla', 'entregado')")
con.close()

codigo, salida = dbt("build", "--select", "stg_pedidos+")
print(*[l for l in salida if "FAIL" in l or "ERROR" in l or "SKIP" in l or "Done" in l], sep="\n")
print("\ncódigo de salida:", codigo, "(distinto de 0: un orquestador pararía aquí)")

con = abrir()
con.execute("DELETE FROM plata.pedidos WHERE id_linea = 999999999")
print("Fila mala eliminada; el almacén vuelve a estar limpio.")

## 7.6 · Spark: la misma idea, repartida en un clúster

Spark reparte los datos en **particiones** entre varias máquinas y ejecuta tu plan sobre cada una. Su API de DataFrame se parece mucho a las que ya conoces. Para practicar sin montar nada:
- **Google Colab:** `pip install pyspark` y `SparkSession.builder.master("local[*]")` (Colab ya trae Java).
- **Databricks Free Edition:** cuenta gratuita en la nube con Spark ya configurado (comprueba el nombre y los límites vigentes: cambian).

Esqueleto para Colab o Databricks (no se ejecuta aquí):

```python
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").appName("tiendasur").getOrCreate()
df = spark.read.parquet("pedidos_parquet")            # o una ruta en el almacenamiento de Databricks
(df.filter(F.col("estado") == "entregado")
   .groupBy("mes", "canal")
   .agg(F.sum(F.col("cantidad") * F.col("precio_unitario")).alias("ingresos"))
   .orderBy("mes")
   .write.mode("overwrite").partitionBy("mes").parquet("oro/ventas_mes_canal"))
```

In [ ]:
# ¿Tenemos Spark en este equipo? Necesita PySpark y un Java instalado.
hay_pyspark = importlib.util.find_spec("pyspark") is not None
try:                                   # en macOS "java" puede existir como un aviso vacío: probamos de verdad
    hay_java = subprocess.run(["java", "-version"], capture_output=True).returncode == 0
except FileNotFoundError:
    hay_java = False
print(f"PySpark instalado: {hay_pyspark} · Java disponible: {hay_java}")

if hay_pyspark and hay_java:
    from pyspark.sql import SparkSession, functions as F
    spark = SparkSession.builder.master("local[2]").appName("tiendasur").getOrCreate()
    sdf = spark.read.parquet(str(RUTA_PART))
    (sdf.filter(F.col("estado") == "entregado").groupBy("mes")
        .agg(F.sum(F.col("cantidad") * F.col("precio_unitario")).alias("ingresos")).orderBy("mes").show(3))
    spark.stop()
else:
    print("Sin Spark local: no pasa nada. El esqueleto para Colab/Databricks está en el apartado siguiente,")
    print("y abajo simulamos lo que Spark hace por dentro para que entiendas por qué escala.")

# Equivalencias de la API de DataFrame (la misma idea en cuatro sintaxis)
equivalencias = pd.DataFrame([
    ("Filtrar",     'df[df.estado == "entregado"]',          'df.filter(pl.col("estado") == "entregado")', 'df.filter(F.col("estado") == "entregado")',  "WHERE estado = 'entregado'"),
    ("Columna nueva", 'df.assign(importe=df.q * df.p)',       'df.with_columns((pl.col("q")*pl.col("p")).alias("importe"))', 'df.withColumn("importe", F.col("q")*F.col("p"))', "SELECT *, q*p AS importe"),
    ("Agrupar",     'df.groupby("mes").agg(...)',             'df.group_by("mes").agg(...)',                'df.groupBy("mes").agg(...)',                  "GROUP BY mes"),
    ("Unir",        'df.merge(dim, on="id")',                 'df.join(dim, on="id")',                      'df.join(dim, on="id")',                       "JOIN dim USING (id)"),
    ("Ordenar",     'df.sort_values("x")',                    'df.sort("x")',                               'df.orderBy("x")',                             "ORDER BY x"),
], columns=["operación", "pandas", "polars", "pyspark", "SQL (DuckDB/Spark SQL)"])
print(equivalencias.to_string(index=False))

In [ ]:
# Qué hace Spark por dentro, en 20 líneas: partir los datos, agregar cada trozo, repartir por clave, combinar.
def agregacion_distribuida(df, n_particiones=8):
    # 1) Los datos se dividen en particiones (en un clúster, cada una en una máquina distinta).
    trozo = len(df) // n_particiones + 1
    particiones = [df.iloc[i:i + trozo][["mes", "cantidad", "precio_unitario"]] for i in range(0, len(df), trozo)]
    # 2) MAP + agregación parcial dentro de cada partición (no hay que mover las 4 millones de filas).
    parciales = [p.assign(importe=p.cantidad * p.precio_unitario).groupby("mes")["importe"].sum() for p in particiones]
    # 3) SHUFFLE: cada mes se envía a un único "reductor" (aquí, simplemente concatenamos y reagrupamos).
    # 4) REDUCE: combinamos los resultados parciales.
    return pd.concat(parciales).groupby(level=0).sum()

distribuido = agregacion_distribuida(pedidos)
directo = (pedidos["cantidad"] * pedidos["precio_unitario"]).groupby(pedidos["mes"]).sum()
assert np.allclose(distribuido.sort_index(), directo.sort_index())
print("Resultado distribuido == resultado directo.")
print(f"En el shuffle viajan como mucho {8 * distribuido.shape[0]} sumas parciales, no {len(pedidos):,} filas.")

## 7.7 · Orquestación: DAGs, reintentos, idempotencia

Un **DAG** (grafo acíclico dirigido) describe qué tareas hay y de cuáles depende cada una. Un orquestador (Airflow, Dagster, Prefect) lo ejecuta según un calendario, **reintenta** lo que falla, deja un historial y permite el *backfill* (reprocesar días pasados). Todo eso solo es seguro si las tareas son **idempotentes**, que es lo que hicimos en las capas.

In [ ]:
from graphlib import TopologicalSorter

# Un DAG es un diccionario: tarea -> tareas de las que depende.
DEPENDENCIAS = {
    "extraer":  set(),
    "bronce":   {"extraer"},
    "plata":    {"bronce"},
    "oro":      {"plata"},
    "calidad":  {"plata"},
    "informe":  {"oro", "calidad"},
}
print("Orden de ejecución:", list(TopologicalSorter(DEPENDENCIAS).static_order()))

con = abrir()
con.register("productos", productos)
FALLOS_SIMULADOS = {("extraer", "2026-09-05"): 1}      # la extracción del día 5 falla una vez (red caída)

def ejecutar_dag(fecha, reintentos=2):
    contexto = {}
    tareas = {
        "extraer": lambda: contexto.update(lote=lote_del_dia(fecha)),
        "bronce":  lambda: cargar_bronce(con, fecha, contexto["lote"]),
        "plata":   lambda: limpiar_plata(con, fecha),
        "oro":     lambda: agregar_oro(con, fecha),
        "calidad": lambda: contexto.update(tasa=validar_calidad(con, fecha)),
        "informe": lambda: print(f"  {fecha}: ok · rechazo {contexto['tasa']:.2%}"),
    }
    for nombre in TopologicalSorter(DEPENDENCIAS).static_order():
        for intento in range(1, reintentos + 2):
            try:
                if FALLOS_SIMULADOS.get((nombre, fecha), 0) > 0:
                    FALLOS_SIMULADOS[(nombre, fecha)] -= 1
                    raise ConnectionError("API de pedidos no responde")
                tareas[nombre]()
                break
            except ConnectionError as e:
                print(f"  {fecha}/{nombre}: intento {intento} falló ({e})")
                if intento > reintentos:
                    raise
                time.sleep(0.1 * 2 ** (intento - 1))          # espera exponencial entre reintentos

# "Backfill": reprocesar un rango de fechas. Funciona porque cada tarea es idempotente.
for f in pd.date_range("2026-09-04", "2026-09-06").strftime("%Y-%m-%d"):
    ejecutar_dag(f)
print(con.sql("SELECT fecha, COUNT(*) AS filas, ROUND(SUM(ingresos)) AS ingresos FROM oro.fct_ventas_dia GROUP BY fecha ORDER BY fecha").df())

In [ ]:
(BASE / "dags").mkdir(exist_ok=True)
(BASE / "dags" / "tiendasur_pedidos_diarios.py").write_text('''"""DAG de Airflow para el pipeline diario de pedidos de TiendaSur.

Sintaxis de Airflow 3 (airflow.sdk). En Airflow 2.x los decoradores se importan de
airflow.decorators: comprueba la versión que tengas y su documentación.
"""
from datetime import datetime, timedelta

from airflow.sdk import dag, task


@dag(
    dag_id="tiendasur_pedidos_diarios",
    schedule="@daily",
    start_date=datetime(2026, 9, 1),
    catchup=True,                      # permite el backfill de días pasados
    max_active_runs=1,
    default_args={"retries": 2, "retry_delay": timedelta(minutes=5)},
    tags=["tiendasur", "pedidos"],
)
def pedidos_diarios():
    @task
    def extraer(ds=None) -> str:
        ...  # descarga el extracto del día `ds` y devuelve la ruta

    @task
    def cargar_bronce(ruta: str, ds=None): ...

    @task
    def limpiar_plata(ds=None): ...

    @task
    def agregar_oro(ds=None): ...

    @task
    def validar_calidad(ds=None): ...   # lanza excepción si el rechazo supera el umbral

    ruta = extraer()
    b = cargar_bronce(ruta)
    p = limpiar_plata()
    o = agregar_oro()
    c = validar_calidad()
    b >> p >> [o, c]


pedidos_diarios()
''')
import py_compile
py_compile.compile(str(BASE / "dags" / "tiendasur_pedidos_diarios.py"), doraise=True)
print("DAG escrito y con sintaxis correcta (Airflow no está instalado: es un archivo de ejemplo, no se ejecuta aquí).")

## 7.8 · Streaming: datos que no paran de llegar

Kafka es un registro distribuido de eventos: los productores escriben en *topics* (divididos en particiones) y los consumidores leen a su ritmo, apuntando hasta dónde han leído (*offset*). Garantiza entrega **al menos una vez**: pueden llegar duplicados. Los eventos además llegan **desordenados**: lo que ocurrió a las 10:04 puede llegar a las 10:09.

Para agregarlos se usan **ventanas** (aquí, de 5 minutos) y una **marca de agua** (*watermark*): «ya no espero eventos de antes de este instante». Cuando la marca de agua pasa el final de una ventana, se cierra y se emite su resultado. Un evento que llega después de cerrada es un **tardío**. Simulamos el consumidor completo:

In [ ]:
# Simulamos una hora de pedidos de la web de TiendaSur como un flujo de eventos (como en un topic de Kafka).
rng_s = np.random.default_rng(3)
n_ev = 20_000
t_evento = np.sort(rng_s.uniform(0, 3600, n_ev))                          # segundos desde las 10:00
retraso = rng_s.exponential(15, n_ev)                                      # latencia normal: ~15 s
retraso[rng_s.random(n_ev) < 0.03] += 240                                  # 3 % llega con 4 minutos de retraso
eventos = [{"id": i, "t": float(t_evento[i]), "llegada": float(t_evento[i] + retraso[i]),
            "importe": float(rng_s.gamma(3, 12))} for i in range(n_ev)]
eventos += [dict(eventos[i]) for i in rng_s.integers(0, n_ev, 200)]        # reentregas (Kafka: at-least-once)
eventos.sort(key=lambda e: e["llegada"])                                   # el consumidor los ve por orden de LLEGADA
print(f"{len(eventos):,} mensajes recibidos (con {200} duplicados) · el orden de llegada != el orden real")

def consumir(eventos, ventana=300, retraso_permitido=60):
    """Consumidor con ventanas fijas (tumbling) de `ventana` s y marca de agua (watermark)."""
    abiertas = defaultdict(lambda: {"pedidos": 0, "ingresos": 0.0})
    cerradas, tardios, vistos, duplicados, max_t = {}, [], set(), 0, 0.0
    for ev in eventos:
        if ev["id"] in vistos:                      # idempotencia: ya lo habíamos procesado
            duplicados += 1
            continue
        vistos.add(ev["id"])
        k = int(ev["t"] // ventana)
        if k in cerradas:                           # su ventana ya se cerró y emitió: llega tarde
            tardios.append(ev)
            continue
        abiertas[k]["pedidos"] += 1
        abiertas[k]["ingresos"] += ev["importe"]
        max_t = max(max_t, ev["t"])
        marca_agua = max_t - retraso_permitido      # "ya no espero eventos anteriores a esto"
        for kk in [x for x in abiertas if (x + 1) * ventana <= marca_agua]:
            cerradas[kk] = abiertas.pop(kk)
    cerradas.update(abiertas)                       # al terminar, volcamos lo que quede abierto
    return cerradas, tardios, duplicados

ventanas, tardios, dup = consumir(eventos)
procesados = sum(v["pedidos"] for v in ventanas.values())
assert procesados + len(tardios) + dup == len(eventos)                    # nada se pierde sin contarlo
print(f"procesados {procesados:,} · tardíos {len(tardios)} · duplicados descartados {dup}")
resumen = pd.DataFrame(ventanas).T.sort_index()
resumen.index = [f"{int(k * 5):02d}-{int(k * 5 + 5):02d} min" for k in resumen.index]
print(resumen.round(0).head(6))

In [ ]:
(BASE / "streaming").mkdir(exist_ok=True)
(BASE / "streaming" / "consumidor_kafka.py").write_text('''"""Esqueleto de un consumidor Kafka real (librería confluent-kafka). No se ejecuta en el curso."""
import json

from confluent_kafka import Consumer

consumidor = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "ventas-ventanas",        # los consumidores del mismo grupo se reparten las particiones
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,          # confirmamos el offset DESPUÉS de procesar (at-least-once)
})
consumidor.subscribe(["pedidos.v1"])

try:
    while True:
        msg = consumidor.poll(1.0)
        if msg is None or msg.error():
            continue
        evento = json.loads(msg.value())
        # ... aquí iría la misma lógica de consumir(): dedup por id, ventana, marca de agua ...
        consumidor.commit(message=msg)    # offset confirmado: si reiniciamos, seguimos desde aquí
finally:
    consumidor.close()
''')
import py_compile
py_compile.compile(str(BASE / "streaming" / "consumidor_kafka.py"), doraise=True)
print("Esqueleto de consumidor Kafka escrito (sintaxis correcta; no hay broker en el curso).")

## Qué has construido

Un mini-almacén con arquitectura medallón, un proyecto dbt con tests, un orquestador mínimo con reintentos y backfill, y un consumidor de streaming con ventanas. Súbelo a GitHub con un README que cuente el caso y las decisiones (**pieza de portafolio** para perfiles de ML Engineer / Data Engineer).

## Ejercicios

Los ejercicios usan los objetos creados arriba (`pedidos`, `productos`, `RUTA_PART`, `BASE`, `abrir()`, las funciones del pipeline…). Ejecuta el notebook entero antes de empezar.

1. **Benchmark propio.** Calcula el *top 10 de productos por ingresos* en el último trimestre (`mes >= '2026-07'`, solo entregados) con pandas, Polars y DuckDB. Comprueba que los tres dan el mismo top 10 y cronométralos. ¿Quién gana y por qué?
2. **Compresión de Parquet.** Escribe la muestra de 500.000 filas con compresión `none`, `snappy` y `zstd`. Tabula tamaño en disco y tiempo de escritura y de lectura. ¿Cuál elegirías para un almacén que se lee mucho y se escribe una vez al día?
3. **dbt.** Añade un modelo `fct_ventas_mes` (ingresos y unidades por mes y categoría, uniendo con `dim_producto`) y un test `accepted_values` o un test singular sobre él. Cierra antes tu conexión a DuckDB (`con.close()`), ejecútalo con `dbt build` y comprueba que la suma de `ingresos` coincide con la de `fct_ventas_dia`.
4. **Puerta de calidad.** Fuerza un lote con un 12 % de precios nulos y comprueba que `validar_calidad` detiene el pipeline para ese día **sin dejar datos a medias en oro**. Pista: ¿en qué orden están las tareas? Reordena o protege el pipeline para que oro solo se escriba si la calidad pasa.
5. **Streaming.** Repite el consumidor con `retraso_permitido` de 0, 60, 180 y 300 segundos. Tabula el porcentaje de eventos descartados por tardíos y cuántos segundos más tarda de media en cerrarse una ventana. Explica el compromiso entre exactitud y latencia.

In [ ]:
# Ejercicio 1 · tu código aquí


In [ ]:
# Ejercicio 2 · tu código aquí


In [ ]:
# Ejercicio 3 · tu código aquí


In [ ]:
# Ejercicio 4 · tu código aquí


In [ ]:
# Ejercicio 5 · tu código aquí
